In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "paths.py").exists())
sys.path.insert(0, str(ROOT))
from paths import *


## GPU and python


In [ ]:
# OneTrainer on Colab, SDXL.
# run the setup cells once per session, then the train cell.
#
# two things that broke this before: a second OneTrainer clone in /content
# shadowing the Drive repo, and installing diffusers from master instead of
# the pinned commit. don't re-add a cell that clones either into /content.
!nvidia-smi

import sys
print("python:", sys.version)
if sys.version_info < (3, 10) or sys.version_info >= (3, 13):
    print("onetrainer wants python >=3.10,<3.13")
else:
    print("python ok")


Thu Aug 27 00:07:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             54W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Mount, update, clear stray clones


In [ ]:
import os

PROJECT = str(ROOT)
REPO = f'{PROJECT}/OneTrainer'

os.makedirs(PROJECT, exist_ok=True)

if not os.path.exists(REPO):
    %cd $PROJECT
    !git clone --recursive https://github.com/Nerogar/OneTrainer.git

%cd $REPO

# stray copies shadow the Drive repo and cause version-mismatch import errors
!rm -rf /content/OneTrainer /content/diffusers

!git log -1 --format="OneTrainer commit: %h  %cd"

print("cwd:", os.getcwd())


Mounted at /content/drive
/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer
OneTrainer commit: 9a270603  Sun Feb 8 09:18:39 2026 +0100
Active dir: /content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer


## Install pinned requirements


In [ ]:
# exact versions this commit wants, pinned diffusers included. restart the
# runtime after (Runtime > Restart session), re-run the mount cell
%cd {ROOT}/OneTrainer
!pip install -r requirements.txt


/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu128
Ignoring triton-windows: markers 'sys_platform == "win32"' don't match your environment
Obtaining diffusers from git+https://github.com/huggingface/diffusers.git@6a1904e#egg=diffusers (from -r requirements-global.txt (line 23))
  Updating ./src/diffusers clone (to revision 6a1904e)
  Running command git fetch -q --tags
  Running command git reset --hard -q 6a1904e
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
Obtaining mgds from git+https://github.com/Nerogar/mgds.git@a0c84a3#egg=mgds (from -r requirements-global.txt (line 35))
  Updating ./src/mgds clone (to revision a0c84a3)
  Running command git fetch -q --tags
  Running command git reset --hard -q a0c84a3
  Installing build dependen

## Verify


In [ ]:
!python -c "import torch, torchvision, transformers, diffusers; print('torch', torch.__version__); print('torchvision', torchvision.__version__); print('transformers', transformers.__version__); print('diffusers', diffusers.__version__)"
!python -c "import torch; print('cuda:', torch.cuda.is_available())"
!python -c "from torchvision.io import write_video; print('write_video OK')"
!python -c "import mgds; print('mgds:', mgds.__file__)"


torch 2.8.0+cu128
torchvision 0.23.0+cu128
transformers 4.56.2
diffusers 0.37.0.dev0
cuda: True
write_video OK
mgds: None


## Check the config


In [ ]:
# catches the failures that give a silent no-op run: empty concepts list,
# relative paths resolving somewhere else, missing image folderstensorboard on (colab has no /usr/bin/tensorboard)
import json, os

CONFIG = str(ROOT / "pixartsigma_colab.json")
REPO = str(ROOT / "OneTrainer")

c = json.load(open(CONFIG))

for k in ["base_model_name", "model_type", "training_method", "peft_type",
          "train_device", "temp_device", "resolution", "epochs", "batch_size",
          "learning_rate", "save_every", "save_every_unit",
          "workspace_dir", "output_model_destination"]:
    print(f"{k}: {c.get(k)}")

print()
problems = []

if c.get("concepts") == []:
    problems.append("concepts is [] -- must be null, or the concept file is ignored "
                    "and training silently runs on zero images")
if c.get("train_device") != "cuda":
    problems.append(f"train_device is {c.get('train_device')!r} -- should be 'cuda'")
if c.get("tensorboard"):
    problems.append("tensorboard is true -- Colab has no /usr/bin/tensorboard, set false")

cf = c.get("concept_file_name")
if cf:
    resolved = cf if os.path.isabs(cf) else os.path.join(REPO, cf)
    print("concept file:", resolved, "| exists:", os.path.exists(resolved))
    if os.path.exists(resolved):
        for con in json.load(open(resolved)):
            d = con.get("path")
            n = len(os.listdir(d)) if d and os.path.isdir(d) else 0
            print(f"  concept path: {d} | files: {n}")
            if n == 0:
                problems.append(f"concept path has no files: {d}")
    else:
        problems.append("concept file not found")

print()
if problems:
    for p in problems:
        print("PROBLEM:", p)
else:
    print("config looks OK")


base_model_name: PixArt-alpha/PixArt-Sigma-XL-2-1024-MS
model_type: PIXART_SIGMA
training_method: LORA
peft_type: LORA
train_device: cuda
temp_device: cpu
resolution: 1024
epochs: 100
batch_size: 4
learning_rate: 0.0001
save_every: 0
save_every_unit: NEVER
workspace_dir: /content/drive/MyDrive/Synthetic_Plants_Project/workspace/pixart_achillea_run
output_model_destination: /content/drive/MyDrive/Synthetic_Plants_Project/outputs/pixart_achillea/lora.safetensors

concept file: /content/drive/MyDrive/Synthetic_Plants_Project/Notebooks/OneTrainer/modelconfigs/train_concepts.json | exists: True
  concept path: /content/drive/MyDrive/Synthetic_Plants_Project/Datasets/Achillea_Maritima_2 | files: 276

config looks OK


## Train


In [ ]:
# the shim only patches torchvision.io.write_video, which some builds no
# longer export. if the verify cell said write_video OK, skip both shim
# cells and call scripts/train.py directly. /content is wiped on restart,
# so re-run the writefile cell each session.
%%writefile /content/shim.py
import sys, os, runpy
import torchvision.io
if not hasattr(torchvision.io, "write_video"):
    torchvision.io.write_video = lambda *a, **k: None

root = os.getcwd()
sys.path.insert(0, os.path.join(root, "scripts"))
sys.path.insert(0, root)

sys.argv = sys.argv[1:]
runpy.run_path(os.path.join(root, "scripts", "train.py"), run_name="__main__")


Writing /content/shim.py


In [ ]:
!find {ROOT} -name "*.jpg" -newermt "-1 hour" 2>/dev/null | head -20
!find /content -name "*.jpg" -newermt "-1 hour" -not -path "*/drive/*" 2>/dev/null | head -20


In [ ]:
!ls -la {ROOT}/training/concepts
!ls -la {ROOT}/training/concepts/pixart/ 2>/dev/null
!ls -la {ROOT}/training/concepts/pixart


total 31
drwx------ 2 root root 4096 Aug 25 16:59 flux_lora
-rw------- 1 root root 9427 Aug 26 20:51 generate_eval.py
-rw------- 1 root root 5170 Aug 26 19:33 Label.ipynb
drwx------ 2 root root 4096 Aug 25 16:59 pixart_lora
drwx------ 2 root root 4096 Aug 25 16:58 qwen_lora
drwx------ 2 root root 4096 Aug 25 16:59 sdxl_lora
total 56
-rw------- 1 root root 14980 Aug 25 16:56 pixartsigma_colab_achillea.json
-rw------- 1 root root 14995 Aug 25 16:56 pixartsigma_colab_carpobrotus.json
-rw------- 1 root root 14980 Aug 25 16:56 pixartsigma_colab_eryngium.json
-rw------- 1 root root  2254 Aug 25 16:56 train_concepts_achillea.json
-rw------- 1 root root  2270 Aug 25 16:56 train_concepts_carpobrotus.json
-rw------- 1 root root  2256 Aug 25 16:56 train_concepts_eryngium.json
-rw------- 1 root root   832 Aug 25 16:56 train_samples_achillea.json
-rw------- 1 root root   832 Aug 25 16:56 train_samples_carpobrotus.json
-rw------- 1 root root   832 Aug 25 16:56 train_samples_eryngium.json


In [ ]:
%cd {ROOT}/OneTrainer
!python -u /content/shim.py train.py --config-path {ROOT}/training/configs/sdxl/sdxl_colab_achillea.json


Streaming output truncated to the last 5000 lines.
epoch:  26% 26/100 [24:13<1:03:09, 51.21s/it]
step:   0% 0/32 [00:00<?, ?it/s]
step:   0% 0/32 [00:01<?, ?it/s, loss=0.255, smooth loss=0.18]
step:   3% 1/32 [00:01<00:48,  1.56s/it, loss=0.255, smooth loss=0.18]
step:   3% 1/32 [00:03<00:48,  1.56s/it, loss=0.311, smooth loss=0.181]
step:   6% 2/32 [00:03<00:45,  1.52s/it, loss=0.311, smooth loss=0.181]
step:   6% 2/32 [00:04<00:45,  1.52s/it, loss=0.137, smooth loss=0.181]
step:   9% 3/32 [00:04<00:44,  1.53s/it, loss=0.137, smooth loss=0.181]
step:   9% 3/32 [00:06<00:44,  1.53s/it, loss=0.219, smooth loss=0.181]
step:  12% 4/32 [00:06<00:42,  1.53s/it, loss=0.219, smooth loss=0.181]
step:  12% 4/32 [00:07<00:42,  1.53s/it, loss=0.0777, smooth loss=0.18]
step:  16% 5/32 [00:07<00:41,  1.53s/it, loss=0.0777, smooth loss=0.18]
step:  16% 5/32 [00:09<00:41,  1.53s/it, loss=0.117, smooth loss=0.179]
step:  19% 6/32 [00:09<00:39,  1.52s/it, loss=0.117, smooth loss=0.179]
step:  19% 6/32 

In [ ]:
%cd {ROOT}/OneTrainer
!python -u /content/shim.py train.py --config-path {ROOT}/training/configs/sdxl/sdxl_colab_eryngium.json


/content/drive/MyDrive/Synthetic_Plants_Project/OneTrainer
2026-08-27 17:35:33.553103: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-27 17:35:33.625794: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
tokenizer_config.

In [4]:
%cd {ROOT}/OneTrainer
!python -u /content/shim.py train.py --config-path {ROOT}/training/configs/sdxl/sdxl_colab_carpobrotus.json



step:   0% 0/16 [00:00<?, ?it/s]
step:   0% 0/16 [00:01<?, ?it/s, loss=0.126, smooth loss=0.171]
step:   6% 1/16 [00:01<00:23,  1.56s/it, loss=0.126, smooth loss=0.171]
step:   6% 1/16 [00:03<00:23,  1.56s/it, loss=0.275, smooth loss=0.172]
step:  12% 2/16 [00:03<00:21,  1.55s/it, loss=0.275, smooth loss=0.172]

sampling:   0% 0/20 [00:00<?, ?it/s]

sampling:   5% 1/20 [00:00<00:06,  3.02it/s]

sampling:  10% 2/20 [00:00<00:05,  3.28it/s]

sampling:  15% 3/20 [00:00<00:05,  3.38it/s]

sampling:  20% 4/20 [00:01<00:04,  3.44it/s]

sampling:  25% 5/20 [00:01<00:04,  3.45it/s]

sampling:  30% 6/20 [00:01<00:04,  3.46it/s]

sampling:  35% 7/20 [00:02<00:03,  3.45it/s]

sampling:  40% 8/20 [00:02<00:03,  3.47it/s]

sampling:  45% 9/20 [00:02<00:03,  3.49it/s]

sampling:  50% 10/20 [00:02<00:02,  3.50it/s]

sampling:  55% 11/20 [00:03<00:02,  3.50it/s]

sampling:  60% 12/20 [00:03<00:02,  3.51it/s]

sampling:  65% 13/20 [00:03<00:01,  3.51it/s]

sampling:  70% 14/20 [00:04<00:01,  3.50it/s]

In [5]:
%cd {ROOT}
!python Configs_and_Concepts/generate_eval_sdxl.py --all --n 100


/content/drive/MyDrive/Synthetic_Plants_Project
model=sdxl  n=100  species: achillea, eryngium, carpobrotus

sdxl/achillea
  prompt: a detailed realistic photograph of l35tgyhtew
  size:   1024x1024  steps=30  cfg=7.0
  out:    /content/drive/MyDrive/Synthetic_Plants_Project/generated/sdxl_achillea
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
model_index.json: 100% 609/609 [00:00<00:00, 4.71MB/s]
Fetching 19 files:   0% 0/19 [00:00<?, ?it/s]
text_encoder/model.fp16.safetensors:   0% 0.00/246M [00:00<?, ?B/s]

vae_1_0/diffusion_pytorch_model.fp16.saf(…):   0% 0.00/167M [00:00<?, ?B/s]


vae/diffusion_pytorch_model.fp16.safeten(…):   0% 0.00/167M [00:00<?, ?B/s]



unet/diffusion_pytorch_model.fp16.safete(…):   0% 0.00/5.14G [00:00<?, ?B/s

In [7]:
%cd {ROOT}
!python Configs_and_Concepts/generate_evalflux.py --model flux2 --all --n 100


/content/drive/MyDrive/Synthetic_Plants_Project
model=flux2  n=100  species: achillea, eryngium, carpobrotus

flux2/achillea
  prompt: a detailed realistic photograph of l35tgyhtew
  size:   1024x1024  steps=28  cfg=3.5
  out:    /content/drive/MyDrive/Synthetic_Plants_Project/generated/flux2_achillea
2026-08-27 21:46:53.309602: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-27 21:46:53.381557: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We

In [ ]:
from huggingface_hub import login
login()


In [8]:
import json
from huggingface_hub import login
c = json.load(open(str(ROOT / "training/concepts/qwen/qwen_colab_carpobrotus.json")))
login(c["secrets"]["huggingface_token"])


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


LocalTokenNotFoundError: Token is required (`token=True`), but no token found. You need to provide a token or be logged in to Hugging Face with `hf auth login` or `huggingface_hub.login`. See https://huggingface.co/settings/tokens.

In [ ]:
# @title
%cd {ROOT}
!python Configs_and_Concepts/generate_eval.py --species achillea --n 2


/content/drive/MyDrive/Synthetic_Plants_Project
2026-08-26 20:52:04.171909: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-26 20:52:04.243580: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
generating 2 per species: ac

In [ ]:
# @title
from safetensors import safe_open
p = str(ROOT / "outputs/lora/pixart_achillea/lora.safetensors")
with safe_open(p, framework="pt") as f:
    keys = list(f.keys())
print(len(keys), "tensors")
for k in keys[:12]:
    print(" ", k)


861 tensors
  lora_transformer_adaln_single_emb_timestep_embedder_linear_1.alpha
  lora_transformer_adaln_single_emb_timestep_embedder_linear_1.lora_down.weight
  lora_transformer_adaln_single_emb_timestep_embedder_linear_1.lora_up.weight
  lora_transformer_adaln_single_emb_timestep_embedder_linear_2.alpha
  lora_transformer_adaln_single_emb_timestep_embedder_linear_2.lora_down.weight
  lora_transformer_adaln_single_emb_timestep_embedder_linear_2.lora_up.weight
  lora_transformer_adaln_single_linear.alpha
  lora_transformer_adaln_single_linear.lora_down.weight
  lora_transformer_adaln_single_linear.lora_up.weight
  lora_transformer_caption_projection_linear_1.alpha
  lora_transformer_caption_projection_linear_1.lora_down.weight
  lora_transformer_caption_projection_linear_1.lora_up.weight


## Backup Resume


In [ ]:
# @title
# !ls -la {ROOT}/outputs/workspace*/backup/

# %cd {ROOT}/OneTrainer
# !python -u /content/shim.py train.py \
#   --config-path {ROOT}/training/configs/pixart/pixartsigma_colab.json \
#   --resume-from-checkpoint {ROOT}/outputs/workspace<run>/backup/last


## Freeze Environment


In [ ]:
# @title
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H%M")
!pip freeze > {ROOT}/requirements_frozen_{stamp}.txt
print("written: requirements_frozen_" + stamp + ".txt")


written: requirements_frozen_20260825_0018.txt
